# RAG Workshop: Reranking in Retrieval-Augmented Generation

## Overview
This notebook provides a comprehensive guide to **reranking** in Retrieval-Augmented Generation (RAG) systems, including concepts, importance, techniques, and practical code examples.

## 1. What is Reranking?

**Reranking** is a technique used in information retrieval and RAG systems to re-order search results based on relevance scores. 

### Traditional RAG Pipeline (without Reranking)
1. User asks a question
2. Retrieve top-k documents using an embedding model
3. Pass documents to LLM for generation
4. LLM answers based on retrieved documents

### Problem with Traditional Approach
- **Embedding models** (like sentence transformers) may not perfectly capture semantic relevance
- Retrieved documents might include less relevant or noisy documents
- This can lead to **hallucinations** or inaccurate answers

### RAG Pipeline with Reranking
1. User asks a question
2. Retrieve top-k documents (e.g., top-100) using embedding model
3. **Rerank** the documents using a more powerful model
4. Select top reranked documents (e.g., top-5)
5. Pass to LLM for generation

This two-stage retrieval approach significantly improves answer quality!

## 2. Why Reranking Matters: Key Benefits

### 2.1 Improved Relevance
- **Better precision**: Filters out borderline or irrelevant documents
- **Question-aware ranking**: Uses query context for more accurate scoring

### 2.2 Reduced Noise in Context
- Initial retrieval (embedding-based) retrieves many documents efficiently
- Reranking filters to most relevant ones before LLM
- Prevents LLM from confusion due to conflicting information

### 2.3 Cost Optimization
- First retrieval: Fast, cheap embedding model (e.g., sentence-transformers)
- Reranking: More expensive but on smaller set (e.g., top-100 → top-5)
- Better performance without dramatic cost increase

### 2.4 Reduced Hallucinations
- LLM focuses on truly relevant context
- Less contradiction in source documents
- More consistent and factual responses

### 2.5 Benchmark Improvements
| Metric | Without Reranking | With Reranking |
|--------|------------------|----------------|
| nDCG@5 | 0.65 | 0.78 |
| Precision@5 | 0.72 | 0.85 |
| Answer Accuracy | 78% | 92% |

## 3. Reranking Techniques

### 3.1 Cross-Encoder Models
**How it works**: 
- Takes query + document as input
- Outputs relevance score (0-1 or similarity score)
- More powerful than bi-encoders but slower
- Better captures interaction between query and document

**Popular Models**:
- `ms-marco-MiniLM-L-12-v2`: Fast, good accuracy
- `ms-marco-TinyBERT-L-2-v2`: Smaller, suitable for edge
- `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`: Multilingual
- `cross-encoder/qnli-distilroberta-base`: General QA reranker

### 3.2 LLM-based Reranking
**How it works**:
- Prompt LLM to score relevance of documents
- LLM outputs relevance scores or rankings
- Most powerful but expensive

**Pros**: Captures complex relevance, understands context deeply
**Cons**: Slow, expensive (API costs), non-deterministic

### 3.3 BM25 + ML Reranking
**Combination approach**:
1. BM25 for keyword matching (sparse)
2. ML model reranks top-k BM25 results
3. Combines statistical and semantic signals

### 3.4 Hybrid/Ensemble Reranking
**Multi-signal approach**:
- Combine multiple reranking scores
- Weight by relevance type (semantic, keyword, semantic similarity)
- More robust than single method

### 3.5 Contextual Reranking
**Consider document context**:
- Passage-level vs. document-level ranking
- Consider neighboring passages
- Maintain coherence in selected chunks

## 4. Setup & Installation

### Required Libraries
```bash
pip install sentence-transformers rank_bm25 numpy pandas scikit-learn
pip install torch torchvision torchaudio  # for transformer models
```

Optional (for advanced use):
```bash
pip install faiss-cpu  # for fast similarity search
pip install cohere  # for Cohere Reranker API
pip install jina-client  # for Jina Reranker API
```

In [ ]:
# 4.1: Import Required Libraries
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [ ]:
# 4.2: Create Sample Data for Demo

query = "What is the capital of France?"

documents = [
    "Paris is the capital and largest city of France. It is located in the north-central part of France on the Seine River.",
    "France is a country in Western Europe with several overseas regions and territories.",
    "The Eiffel Tower is located in Paris, France and was built for the 1889 World's Fair.",
    "London is the capital of the United Kingdom and is located on the Thames River.",
    "Berlin is the capital and largest city of Germany.",
    "The Louvre Museum in Paris is the world's largest art museum and houses the Mona Lisa.",
    "French cuisine is known worldwide for its sophistication and culinary traditions.",
    "Madrid is the capital of Spain and is the largest city in Spain.",
    "Rome is the capital of Italy and is known as the Eternal City.",
    "The Arc de Triomphe is a famous monument in Paris, France."
]

print(f"Query: '{query}'")
print(f"\nTotal documents: {len(documents)}")
print("\nSample documents:")
for i, doc in enumerate(documents[:3], 1):
    print(f"  {i}. {doc[:80]}...")

Query: 'What is the capital of France?'

Total documents: 10

Sample documents:
  1. Paris is the capital and largest city of France. It is located in the north-cent...
  2. France is a country in Western Europe with several overseas regions and territor...
  3. The Eiffel Tower is located in Paris, France and was built for the 1889 World's ...


## 5. Code Example 1: Semantic Retrieval (Without Reranking)

In [3]:
# 5.1: Load Embedding Model
print("Loading embedding model (this may take a moment)...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Model loaded successfully\n")

# Encode query and documents
query_embedding = embed_model.encode(query)
doc_embeddings = embed_model.encode(documents)

# Calculate similarity scores
similarities = np.dot(doc_embeddings, query_embedding) / (
    np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
)

# Get top-5 results
top_k = 5
top_indices = np.argsort(similarities)[::-1][:top_k]

print("=" * 80)
print("WITHOUT RERANKING - Semantic Search Results")
print("=" * 80)
print(f"\nTop {top_k} documents by embedding similarity:\n")

results_without_reranking = []
for rank, idx in enumerate(top_indices, 1):
    score = similarities[idx]
    results_without_reranking.append({
        'rank': rank,
        'index': idx,
        'score': score,
        'document': documents[idx]
    })
    print(f"Rank {rank} (Score: {score:.4f})")
    print(f"Doc {idx}: {documents[idx][:70]}...")
    print()

Loading embedding model (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Model loaded successfully

WITHOUT RERANKING - Semantic Search Results

Top 5 documents by embedding similarity:

Rank 1 (Score: 0.7178)
Doc 0: Paris is the capital and largest city of France. It is located in the ...

Rank 2 (Score: 0.6255)
Doc 1: France is a country in Western Europe with several overseas regions an...

Rank 3 (Score: 0.4247)
Doc 6: French cuisine is known worldwide for its sophistication and culinary ...

Rank 4 (Score: 0.3587)
Doc 8: Rome is the capital of Italy and is known as the Eternal City....

Rank 5 (Score: 0.3563)
Doc 9: The Arc de Triomphe is a famous monument in Paris, France....



## 6. Code Example 2: Cross-Encoder Reranking

A cross-encoder is the most effective reranking method. It takes both query and document as input and outputs a relevance score.

In [4]:
# 6.1: Load Cross-Encoder Model and Rerank
print("Loading cross-encoder model (this may take a moment)...")
cross_encoder = CrossEncoder('ms-marco-MiniLM-L-12-v2')
print("✓ Cross-encoder model loaded\n")

# Step 1: Initial retrieval with embedding model (get top-100 candidates)
# (In our case, we'll use all documents as candidates for demo)
candidates = list(range(len(documents)))

# Step 2: Rerank candidates with cross-encoder
query_doc_pairs = [(query, documents[i]) for i in candidates]
reranking_scores = cross_encoder.predict(query_doc_pairs)

# Get top-5 after reranking
top_reranked_indices = np.argsort(reranking_scores)[::-1][:top_k]

print("=" * 80)
print("WITH RERANKING - Cross-Encoder Results")
print("=" * 80)
print(f"\nTop {top_k} documents after cross-encoder reranking:\n")

results_with_reranking = []
for rank, idx in enumerate(top_reranked_indices, 1):
    score = reranking_scores[idx]
    results_with_reranking.append({
        'rank': rank,
        'index': idx,
        'score': score,
        'document': documents[idx]
    })
    print(f"Rank {rank} (Score: {score:.4f})")
    print(f"Doc {idx}: {documents[idx][:70]}...")
    print()

Loading cross-encoder model (this may take a moment)...


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✓ Cross-encoder model loaded

WITH RERANKING - Cross-Encoder Results

Top 5 documents after cross-encoder reranking:

Rank 1 (Score: 8.4433)
Doc 0: Paris is the capital and largest city of France. It is located in the ...

Rank 2 (Score: -0.5103)
Doc 1: France is a country in Western Europe with several overseas regions an...

Rank 3 (Score: -2.7938)
Doc 7: Madrid is the capital of Spain and is the largest city in Spain....

Rank 4 (Score: -3.0318)
Doc 3: London is the capital of the United Kingdom and is located on the Tham...

Rank 5 (Score: -3.0679)
Doc 4: Berlin is the capital and largest city of Germany....



In [5]:
# 6.2: Compare Results
print("=" * 80)
print("COMPARISON: With vs Without Reranking")
print("=" * 80)

comparison_data = []
for rank in range(1, top_k + 1):
    without = results_without_reranking[rank - 1]
    with_reranking = results_with_reranking[rank - 1]
    
    comparison_data.append({
        'Rank': rank,
        'Without Reranking (Embedding)': f"Doc {without['index']} ({without['score']:.4f})",
        'With Reranking (Cross-Encoder)': f"Doc {with_reranking['index']} ({with_reranking['score']:.4f})",
        'Match': '✓' if without['index'] == with_reranking['index'] else '✗'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + comparison_df.to_string(index=False))

# Show which documents changed position
changed_indices = set([r['index'] for r in results_without_reranking]) ^ set([r['index'] for r in results_with_reranking])
if changed_indices:
    print(f"\n📊 Documents with changed rankings: {changed_indices}")
else:
    print("\n✓ Same documents in top-5 (different order)")

COMPARISON: With vs Without Reranking

 Rank Without Reranking (Embedding) With Reranking (Cross-Encoder) Match
    1                Doc 0 (0.7178)                 Doc 0 (8.4433)     ✓
    2                Doc 1 (0.6255)                Doc 1 (-0.5103)     ✓
    3                Doc 6 (0.4247)                Doc 7 (-2.7938)     ✗
    4                Doc 8 (0.3587)                Doc 3 (-3.0318)     ✗
    5                Doc 9 (0.3563)                Doc 4 (-3.0679)     ✗

📊 Documents with changed rankings: {np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(9)}


## 7. Code Example 3: BM25 Keyword-Based Reranking

BM25 is a keyword-based ranking algorithm. It works well for queries with specific terms but misses semantic relationships.

In [17]:
# 7.1: BM25 Reranking Implementation

# Tokenize documents for BM25
def tokenize(text):
    return text.lower().split()

tokenized_docs = [tokenize(doc) for doc in documents]
print("\n✓ Documents tokenized for BM25")
print(f"Tokenized documents: {tokenized_docs[:3]}...")  # Show first 3 tokenized docs for verification

# Initialize BM25
bm25 = BM25Okapi(tokenized_docs)

# Get BM25 scores for query
query_tokens = tokenize(query)
print(f"\nQuery tokens for BM25: {query_tokens}")
bm25_scores = bm25.get_scores(query_tokens)
print(f"BM25 scores: {bm25_scores}")

# Get top-5 by BM25
top_bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]

print("=" * 80)
print("KEYWORD-BASED RANKING - BM25 Results")
print("=" * 80)
print(f"\nTop {top_k} documents by BM25 keyword matching:\n")

for rank, idx in enumerate(top_bm25_indices, 1):
    score = bm25_scores[idx]
    print(f"Rank {rank} (BM25 Score: {score:.4f})")
    print(f"Doc {idx}: {documents[idx][:70]}...")
    print()


✓ Documents tokenized for BM25
Tokenized documents: [['paris', 'is', 'the', 'capital', 'and', 'largest', 'city', 'of', 'france.', 'it', 'is', 'located', 'in', 'the', 'north-central', 'part', 'of', 'france', 'on', 'the', 'seine', 'river.'], ['france', 'is', 'a', 'country', 'in', 'western', 'europe', 'with', 'several', 'overseas', 'regions', 'and', 'territories.'], ['the', 'eiffel', 'tower', 'is', 'located', 'in', 'paris,', 'france', 'and', 'was', 'built', 'for', 'the', '1889', "world's", 'fair.']]...

Query tokens for BM25: ['what', 'is', 'the', 'capital', 'of', 'france?']
BM25 scores: [0.95799046 0.37124291 0.82853076 1.09117829 0.85676324 0.91632945
 0.39777079 1.05167632 1.05167632 0.79554157]
KEYWORD-BASED RANKING - BM25 Results

Top 5 documents by BM25 keyword matching:

Rank 1 (BM25 Score: 1.0912)
Doc 3: London is the capital of the United Kingdom and is located on the Tham...

Rank 2 (BM25 Score: 1.0517)
Doc 8: Rome is the capital of Italy and is known as the Eternal City....

R

## 8. Code Example 4: Production RAG Pipeline with Reranking

Here's a complete, production-ready RAG pipeline that combines:
1. **Retrieval**: Embedding-based semantic search (fast)
2. **Reranking**: Cross-encoder scoring (accurate)
3. **Best Practices**: Batching, error handling, extensibility

In [7]:
# 8.1: Production RAG Pipeline Class

class RAGPipeline:
    """
    Production-ready RAG pipeline with two-stage retrieval and reranking
    """
    
    def __init__(self, 
                 embedding_model_name='all-MiniLM-L6-v2',
                 reranker_model_name='ms-marco-MiniLM-L-12-v2',
                 initial_k=10,
                 final_k=3):
        """
        Initialize the RAG pipeline
        
        Args:
            embedding_model_name: Sentence transformer model for initial retrieval
            reranker_model_name: Cross-encoder model for reranking
            initial_k: Number of documents to retrieve initially
            final_k: Number of documents to return after reranking
        """
        print("Initializing RAG Pipeline...")
        self.embed_model = SentenceTransformer(embedding_model_name)
        self.reranker = CrossEncoder(reranker_model_name)
        self.initial_k = initial_k
        self.final_k = final_k
        self.document_embeddings = None
        print("✓ Pipeline initialized successfully")
    
    def index_documents(self, documents):
        """Embed and index documents for fast retrieval"""
        print(f"Indexing {len(documents)} documents...")
        self.documents = documents
        self.document_embeddings = self.embed_model.encode(documents, show_progress_bar=False)
        print(f"✓ Indexed {len(documents)} documents")
    
    def retrieve_and_rerank(self, query, return_metadata=False):
        """
        Two-stage retrieval and reranking
        
        Args:
            query: Search query
            return_metadata: Whether to return scores and rankings
            
        Returns:
            List of top-k documents (and metadata if requested)
        """
        # Stage 1: Semantic Retrieval
        query_embedding = self.embed_model.encode(query)
        similarities = np.dot(self.document_embeddings, query_embedding) / (
            np.linalg.norm(self.document_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        
        # Get initial candidates
        candidate_indices = np.argsort(similarities)[::-1][:self.initial_k]
        
        # Stage 2: Cross-Encoder Reranking
        query_doc_pairs = [(query, self.documents[idx]) for idx in candidate_indices]
        reranking_scores = self.reranker.predict(query_doc_pairs)
        
        # Get final top-k
        final_indices = candidate_indices[np.argsort(reranking_scores)[::-1][:self.final_k]]
        final_scores = reranking_scores[np.argsort(reranking_scores)[::-1][:self.final_k]]
        
        if return_metadata:
            return [
                {
                    'index': idx,
                    'document': self.documents[idx],
                    'reranking_score': score
                }
                for idx, score in zip(final_indices, final_scores)
            ]
        else:
            return [self.documents[idx] for idx in final_indices]

# Initialize pipeline
print("\n" + "="*80)
print("Creating Production RAG Pipeline")
print("="*80)
pipeline = RAGPipeline(initial_k=8, final_k=3)


Creating Production RAG Pipeline
Initializing RAG Pipeline...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Pipeline initialized successfully


In [8]:
# 8.2: Index Documents and Test Pipeline

# Index documents
pipeline.index_documents(documents)

# Retrieve and rerank
print(f"\n\nQuery: '{query}'")
print("\n" + "="*80)
print("RAG Pipeline Results (with Reranking)")
print("="*80)

results = pipeline.retrieve_and_rerank(query, return_metadata=True)

for rank, result in enumerate(results, 1):
    print(f"\n📄 Rank {rank} (Score: {result['reranking_score']:.4f})")
    print(f"   Document index: {result['index']}")
    print(f"   Content: {result['document']}")


Indexing 10 documents...
✓ Indexed 10 documents


Query: 'What is the capital of France?'

RAG Pipeline Results (with Reranking)

📄 Rank 1 (Score: 8.4433)
   Document index: 0
   Content: Paris is the capital and largest city of France. It is located in the north-central part of France on the Seine River.

📄 Rank 2 (Score: -0.5103)
   Document index: 1
   Content: France is a country in Western Europe with several overseas regions and territories.

📄 Rank 3 (Score: -2.7938)
   Document index: 7
   Content: Madrid is the capital of Spain and is the largest city in Spain.


## 9. Advanced: Hybrid Reranking Strategy

Combine multiple reranking signals for more robust results.

In [21]:
# 9.1: Hybrid Reranking - Combine Multiple Signals

def normalize_scores(scores):
    """Normalize scores to [0, 1] range"""
    min_score = np.min(scores)
    max_score = np.max(scores)
    if max_score - min_score == 0:
        return np.ones_like(scores)
    return (scores - min_score) / (max_score - min_score)

def hybrid_rerank(query, documents, initial_k=10, final_k=3,
                  embedding_model=embed_model, 
                  cross_encoder=cross_encoder,
                  bm25_model=bm25,
                  weights={'semantic': 0.3, 'cross_encoder': 0.5, 'bm25': 0.2}):
    """
    Hybrid reranking combining:
    - Semantic similarity (embedding-based)
    - Cross-encoder scores
    - BM25 keyword matching
    """
    
    # 1. Semantic Retrieval
    query_embedding = embedding_model.encode(query)
    doc_embeddings = embedding_model.encode(documents)
    semantic_scores = np.dot(doc_embeddings, query_embedding) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    
    # Get initial candidates
    candidate_indices = np.argsort(semantic_scores)[::-1][:initial_k]
    
    # 2. Cross-Encoder Scores
    query_doc_pairs = [(query, documents[idx]) for idx in candidate_indices]
    cross_encoder_scores = cross_encoder.predict(query_doc_pairs)
    
    # 3. BM25 Scores
    query_tokens = tokenize(query)
    print(f"\nQuery tokens for BM25: {query_tokens}")
    score = bm25_model.get_scores(query_tokens)
    print(f"BM25 scores: {score}")
    bm25_scores = np.array([score[idx] for idx in candidate_indices])
    
    # Normalize all scores
    semantic_norm = normalize_scores(semantic_scores[candidate_indices])
    cross_encoder_norm = normalize_scores(cross_encoder_scores)
    bm25_norm = normalize_scores(bm25_scores)
    
    # Combine scores
    hybrid_scores = (
        weights['semantic'] * semantic_norm +
        weights['cross_encoder'] * cross_encoder_norm +
        weights['bm25'] * bm25_norm
    )
    
    # Get final top-k
    final_indices = candidate_indices[np.argsort(hybrid_scores)[::-1][:final_k]]
    final_scores = hybrid_scores[np.argsort(hybrid_scores)[::-1][:final_k]]
    
    return final_indices, final_scores, {
        'semantic': semantic_norm[np.argsort(hybrid_scores)[::-1][:final_k]],
        'cross_encoder': cross_encoder_norm[np.argsort(hybrid_scores)[::-1][:final_k]],
        'bm25': bm25_norm[np.argsort(hybrid_scores)[::-1][:final_k]]
    }

# Test hybrid reranking
print("\n" + "="*80)
print("HYBRID RERANKING - Combining Multiple Signals")
print("="*80)

final_indices, hybrid_scores, component_scores = hybrid_rerank(
    query, documents, initial_k=8, final_k=3
)

for rank, (idx, score) in enumerate(zip(final_indices, hybrid_scores), 1):
    print(f"\n📊 Rank {rank} (Hybrid Score: {score:.4f})")
    print(f"   Doc {idx}: {documents[idx][:70]}...")
    print(f"   Component scores:")
    print(f"     - Semantic: {component_scores['semantic'][rank-1]:.4f}")
    print(f"     - Cross-encoder: {component_scores['cross_encoder'][rank-1]:.4f}")
    print(f"     - BM25: {component_scores['bm25'][rank-1]:.4f}")


HYBRID RERANKING - Combining Multiple Signals

Query tokens for BM25: ['what', 'is', 'the', 'capital', 'of', 'france?']
BM25 scores: [0.95799046 0.37124291 0.82853076 1.09117829 0.85676324 0.91632945
 0.39777079 1.05167632 1.05167632 0.79554157]

📊 Rank 1 (Hybrid Score: 0.9725)
   Doc 0: Paris is the capital and largest city of France. It is located in the ...
   Component scores:
     - Semantic: 1.0000
     - Cross-encoder: 1.0000
     - BM25: 0.8623

📊 Rank 2 (Hybrid Score: 0.4609)
   Doc 1: France is a country in Western Europe with several overseas regions an...
   Component scores:
     - Semantic: 0.7686
     - Cross-encoder: 0.4605
     - BM25: 0.0000

📊 Rank 3 (Hybrid Score: 0.3893)
   Doc 7: Madrid is the capital of Spain and is the largest city in Spain....
   Component scores:
     - Semantic: 0.0928
     - Cross-encoder: 0.3229
     - BM25: 1.0000


## 10. Production Reranking Services & APIs

### Open-Source Models (Self-Hosted)
| Model | Speed | Accuracy | Best For |
|-------|-------|----------|----------|
| `ms-marco-MiniLM-L-12-v2` | Fast | Good | General purpose, resource-constrained |
| `ms-marco-TinyBERT-L-2-v2` | Very Fast | Fair | Edge devices, low latency |
| `cross-encoder/qnli-distilroberta-base` | Fast | Excellent | Q&A systems |
| `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` | Fast | Good | Multilingual |

### Paid APIs & Services
| Service | Latency | Cost | Best For |
|---------|---------|------|----------|
| **Cohere Rerank** | 50-200ms | $1-2 per million | State-of-the-art accuracy |
| **Jina Reranker** | 100-300ms | $1-3 per million | Enterprise scale |
| **LLM-based Reranking** | 500-2000ms | High | Complex relevance logic |

### When to Use Each Approach
- **Embedding + Cross-Encoder** (Best for RAG): Low cost, good accuracy, fast
- **Embedding + Cohere/Jina API**: When you need SOTA accuracy, can afford API costs
- **Embedding + LLM Reranking**: Complex domain-specific requirements
- **BM25 Only**: Keyword-heavy queries, budget constraints

## 11. Best Practices & Optimization Tips

### 1. **Choose Appropriate Initial Retrieval Count**
```
- Set initial_k based on reranker speed and latency budget
- General rule: 5x-10x the final_k
- Example: Retrieve 50, rerank to 5
```

### 2. **Batch Reranking for Performance**
```
- Rerank in batches instead of one-by-one
- Reduces overhead and speeds up GPU utilization
- Typical batch size: 8-32 query-document pairs
```

### 3. **Cache Embeddings**
```
- Pre-compute document embeddings (one-time cost)
- Reuse for multiple queries
- Saves significant compute time
```

### 4. **Monitor Reranking Effectiveness**
```
- Track metrics: nDCG@k, MRR, Hit Rate
- A/B test with and without reranking
- Measure cost vs. improvement tradeoff
```

### 5. **Handle Edge Cases**
```
- Empty/short documents: May confuse cross-encoder
- Very long documents: Truncate intelligently
- Duplicate documents: Remove before reranking
```

### 6. **Optimize for Latency**
```
- Use smaller models for real-time applications
- Implement async reranking for batch processing
- Consider GPU acceleration for scale
```

### 7. **Fine-tune for Your Domain**
```
- Train cross-encoders on your specific data
- Can dramatically improve accuracy
- Tools: Sentence-transformers library offers fine-tuning
```

### 8. **Combine Multiple Reranking Signals**
```
- Don't rely on single method alone
- Ensemble multiple rerankers for robustness
- Weight by reliability and speed
```

## 12. Practical Examples & Patterns

### Pattern 1: Batch Reranking
```python
# Efficient batch processing
def batch_rerank(queries, documents, batch_size=8):
    results = {}
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        # Process batch
    return results
```

### Pattern 2: Caching Strategy
```python
# Cache embeddings to avoid recomputation
embedding_cache = {}
for doc in documents:
    if doc not in embedding_cache:
        embedding_cache[doc] = embed_model.encode(doc)
```

### Pattern 3: Fallback Reranking
```python
# Use simpler method if primary fails
def smart_rerank(query, docs):
    try:
        return cross_encoder_rerank(query, docs)
    except:
        # Fallback to BM25
        return bm25_rerank(query, docs)
```

### Pattern 4: Threshold-Based Filtering
```python
# Only use documents above confidence threshold
def filtered_rerank(query, docs, threshold=0.5):
    scores = reranker.predict([(query, doc) for doc in docs])
    return [docs[i] for i, score in enumerate(scores) if score > threshold]
```

## 13. Common Pitfalls & Troubleshooting

### ❌ Pitfall 1: Not Reranking with Enough Initial Documents
**Problem**: Relevant doc filtered out before reranking
**Solution**: Retrieve top-100 initially, rerank to top-5
```python
# Bad
top_k = 5
retrieve_and_rerank(query, k=5)  # May miss relevant docs

# Good
retrieve_and_rerank(query, initial_k=50, final_k=5)
```

### ❌ Pitfall 2: Reranker Slower Than Benefit
**Problem**: Reranking takes too long, hurts UX
**Solution**: Use faster models or reduce initial_k
```python
# Bad - rerank 1000 documents every time
rerank(query, 1000_docs)

# Good - rerank smaller set
rerank(query, 20_docs)
```

### ❌ Pitfall 3: Overfitting Reranker Weights
**Problem**: Hybrid weights optimized for one query distribution
**Solution**: Cross-validate on diverse queries
```python
# Bad - tuned for one domain
weights = {'semantic': 0.1, 'cross_encoder': 0.9, 'bm25': 0}

# Good - balanced for multiple domains
weights = {'semantic': 0.3, 'cross_encoder': 0.5, 'bm25': 0.2}
```

### ❌ Pitfall 4: Ignoring Token Limits
**Problem**: Cross-encoder fails on very long documents
**Solution**: Truncate documents intelligently
```python
# Bad
pairs = [(query, doc) for doc in documents]  # Docs may be too long

# Good
max_tokens = 512
pairs = [(query, truncate_doc(doc, max_tokens)) for doc in documents]
```

### ✅ Good Practice Checklist
- [ ] Measure baseline (embedding-only) performance
- [ ] Test reranking improves key metrics (nDCG, MRR)
- [ ] Monitor latency impact
- [ ] Cache embeddings when possible
- [ ] Use batch processing
- [ ] Have fallback strategy
- [ ] Version models and track changes
- [ ] Log relevance scores for monitoring

## 14. Summary & Key Takeaways

### 🎯 What is Reranking?
- **Two-stage retrieval** for better RAG performance
- **Stage 1**: Fast embedding-based retrieval (e.g., top-100)
- **Stage 2**: Accurate reranking with cross-encoder (e.g., top-5)

### 📊 Why It Matters
| Aspect | Impact |
|--------|--------|
| **Accuracy** | +10-20% improvement in answer quality |
| **Cost** | Minimal increase (rerank small set) |
| **Latency** | +50-500ms (acceptable for RAG) |
| **Hallucinations** | Significantly reduced |

### 🛠 Implementation Options
1. **Self-Hosted**: Cross-encoder models (ms-marco, QA models)
   - ✓ Cheap, fast, controllable
   - ✗ Requires GPU, moderate accuracy

2. **API-Based**: Cohere, Jina
   - ✓ SOTA accuracy, no infrastructure
   - ✗ Higher cost, latency

3. **Hybrid**: Combine multiple signals
   - ✓ Robust, balanced performance
   - ✗ Complex to tune

4. **LLM-Based**: Use your LLM for reranking
   - ✓ Domain-aware, can reason
   - ✗ Expensive, slow

### 📋 Quick Start Checklist
- [ ] Install dependencies: `sentence-transformers`
- [ ] Choose embedding model (e.g., `all-MiniLM-L6-v2`)
- [ ] Choose reranker model (e.g., `ms-marco-MiniLM-L-12-v2`)
- [ ] Set initial_k = 5x-10x of final_k
- [ ] Evaluate improvement on your data
- [ ] Monitor latency and cost
- [ ] Consider caching and batching

### 📚 Further Reading
- Sentence Transformers: https://www.sbert.net/
- MTEB Leaderboard: https://huggingface.co/spaces/mteb/leaderboard
- Information Retrieval: https://en.wikipedia.org/wiki/Information_retrieval
- Cross-Encoders vs Bi-Encoders: See Sentence Transformers documentation

## 15. Complete Working Example: End-to-End RAG with Reranking

Here's a minimal, self-contained example you can use as a starting template.

In [ ]:
# 15.1: Minimal Complete RAG Example

import time

# Example knowledge base
KB = [
    "Python is a high-level programming language known for simplicity and readability.",
    "The Python interpreter executes code line by line, making debugging easier.",
    "Java is a compiled language that runs on the JVM virtual machine.",
    "JavaScript is the primary language for web browsers and frontend development.",
    "Machine learning is a subset of artificial intelligence focused on learning from data.",
    "Deep learning uses neural networks with multiple layers to process data.",
    "Natural language processing (NLP) enables computers to understand human language.",
    "Reranking improves search results by re-ordering them based on relevance.",
    "Information retrieval is the process of finding relevant documents in large collections.",
    "Vector databases store embeddings for fast semantic search.",
]

def simple_rag_pipeline(query, documents, top_initial=8, top_final=3):
    """Simple RAG pipeline with reranking"""
    
    print(f"Query: {query}\n")
    
    # Stage 1: Semantic retrieval
    print("Stage 1: Semantic Retrieval")
    query_emb = embed_model.encode(query)
    doc_embs = embed_model.encode(documents)
    scores = np.dot(doc_embs, query_emb) / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(query_emb))
    
    candidates = np.argsort(scores)[::-1][:top_initial]
    print(f"  ✓ Retrieved top-{top_initial} candidates")
    
    # Stage 2: Reranking
    print("Stage 2: Cross-Encoder Reranking")
    pairs = [(query, documents[i]) for i in candidates]
    scores = cross_encoder.predict(pairs)
    final_idx = candidates[np.argsort(scores)[::-1][:top_final]]
    print(f"  ✓ Reranked to top-{top_final}\n")
    
    # Results
    print("Results:")
    for i, idx in enumerate(final_idx, 1):
        print(f"{i}. {documents[idx]}")
    
    return [documents[i] for i in final_idx]

# Run example
print("="*80)
print("MINIMAL RAG EXAMPLE")
print("="*80 + "\n")

results = simple_rag_pipeline("What is Python?", KB, top_initial=6, top_final=2)

## 16. Resources & Next Steps

### 🔗 Recommended Resources
- **Hugging Face Models**: https://huggingface.co/models?task=text-classification
- **Sentence Transformers**: https://www.sbert.net/examples/applications/
- **MTEB Leaderboard**: Evaluate model performance on your tasks
- **Cohere API**: https://cohere.ai/
- **Jina Reranker**: https://jina.ai/

### 📖 Models to Try
**Fast & Lightweight**
- `ms-marco-TinyBERT-L-2-v2` (fast, low memory)
- `all-MiniLM-L6-v2` (good balance)

**High Accuracy**
- `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (multilingual)
- `cross-encoder/qnli-distilroberta-base` (QA-focused)

### 🎓 Next Steps for Your Project
1. **Baseline**: Measure performance without reranking
2. **Experiment**: Try different reranker models
3. **Optimize**: Fine-tune for your domain if needed
4. **Deploy**: Use in production with caching & batching
5. **Monitor**: Track quality metrics and user feedback
6. **Iterate**: Continuously improve based on data

### 💡 Advanced Topics to Explore
- Multi-vector reranking
- Cross-lingual reranking
- Contextual reranking (passage-level vs document-level)
- Active learning for ranking data
- Personalized reranking
- Real-time ranking with dynamic data

### 🚀 Production Deployment Checklist
- [ ] Containerize with Docker
- [ ] Use load balancing for multiple rerankers
- [ ] Implement caching layer (Redis)
- [ ] Monitor model performance drift
- [ ] Set up alerting for quality degradation
- [ ] Version control for model updates
- [ ] Document inference latency requirements
- [ ] Plan for model updates and rollbacks